# trapX LLM Advisory — **`advisory_any_bar.py`** (any date + bar)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chanakya1998begin/rdp/blob/main/advisory_any_bar_colab_external.ipynb)

**External / read-only version** — **no Drive mount**. The script `gdown`-downloads the day folder from the PUBLIC shared folder (read-only) onto this Colab VM and writes the log locally. Works on **any** Google account with zero Drive setup.

Runs the *actual* standalone `advisory_any_bar.py` tool (not a re-implementation). The script, all skills, **and a minimal `project/` package** (the two pure engine functions for the v5 recompute) are **embedded below** (base64) — fully self-contained.

**No NVIDIA key needed** — the verdict runs on the owner's LLM through the collector. Just **Run all**.

## 1. Install deps
No API key needed on this account — the verdict uses the owner's LLM via the collector.

In [ ]:
# gdown==6.1.0: Colab ships an OLD gdown (5.x) whose folder parser can't
# list a >50-item folder; 6.1.0 can (needed to find the day folder).
!pip install -q 'gdown==6.1.0' openai anthropic python-dotenv langgraph langgraph-checkpoint-sqlite

import os
try:
    from google.colab import userdata
    for _k in ('NVIDIA_API_KEY', 'ANTHROPIC_API_KEY'):
        try:
            _v = userdata.get(_k)
        except Exception:
            _v = None
        if _v:
            os.environ[_k] = _v.strip()
except Exception:
    pass

print('LLM      : via owner collector (no NVIDIA key needed on this account)')

## 2. Write the embedded `advisory_any_bar.py` + `skills/` + `project/` to disk
The minimal `project/` package lets the script **recompute the v5 layer** — essential because the recorded jsonl carries no `v5_*` (the live engine logs raw facts; v5 is derived).

In [ ]:
import base64, json, pathlib

SCRIPT_B64 = "..."
SKILLS_B64 = "..."
PROJECT_B64 = "..."

pathlib.Path('advisory_any_bar.py').write_bytes(base64.b64decode(SCRIPT_B64))
skills = json.loads(base64.b64decode(SKILLS_B64).decode('utf-8'))
sd = pathlib.Path('skills'); sd.mkdir(exist_ok=True)
for name, text in skills.items():
    (sd / name).write_text(text, encoding='utf-8')
proj = json.loads(base64.b64decode(PROJECT_B64).decode('utf-8'))
for relpath, text in proj.items():
    p = pathlib.Path(relpath); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding='utf-8')
print('wrote advisory_any_bar.py +', len(skills), 'skills +', len(proj),
      'project files (engine v5 recompute enabled)')

## 3. Choose the bar + backend
`WHEN` is the bar in the script's native `DD-mon, HH:MM` form.

In [ ]:
#@title Parameters { run: "auto" }
WHEN    = "12-Jun, 09:19"  #@param {type:"string"}
BACKEND = "nvidia"         #@param ["nvidia", "anthropic", "auto"]
YEAR    = 2026             #@param {type:"integer"}
MODEL   = ""               #@param {type:"string"}
print('WHEN   =', WHEN)
print('BACKEND=', BACKEND, '  YEAR=', YEAR, '  MODEL=', MODEL or '(default)')

## 4. Resolve the day + collector (no Drive, no NVIDIA key)
**No Drive mount and no NVIDIA key needed.** The script `gdown`-downloads just the one day folder (read-only), runs the verdict using the **owner's** LLM access via the collector (the key is server-side, never here), and the log is then sent to the owner's `external_user_logs/` folder.

In [ ]:
import sys, os, argparse
sys.path.insert(0, '.')
import advisory_any_bar as aab

# Owner's Apps Script endpoint — does BOTH the LLM proxy and the log save.
COLLECTOR_URL = "https://script.google.com/macros/s/AKfycbx8SDCFUaAf_kxKxUOSpOnb0tsbwQ6OS-F72bjer60LQyI1-uoF7edA0XCvzCpBhtLQ/exec"  #@param {type:"string"}

req = aab.parse_request(argparse.Namespace(when=WHEN, date=None,
                                           time=None, year=YEAR))
DAY_FOLDER = req.tmp_dir_name   # e.g. 'gdrive_tmp_jun_12' (created on download)
print(f'date {req.iso_date}  ->  gdown-download {req.mon_dd!r} into {DAY_FOLDER}/')
print('LLM  ->  via owner collector (no NVIDIA key needed on this account)')
print('log  ->  owner external_user_logs/<you>_<date>_<time>.log')

## 5. Run the advisory (verdict via the owner's LLM)
The LLM call is routed through the owner's collector (`TRAPX_LLM_PROXY`), so **you need no NVIDIA key**. The day's data is gdown-downloaded (read-only); `--no-live` is required on Colab.

In [ ]:
import subprocess, sys, shlex

env = dict(os.environ)
if COLLECTOR_URL.startswith('http'):
    env['TRAPX_LLM_PROXY'] = COLLECTOR_URL   # LLM -> owner's key, server-side
cmd = [sys.executable, 'advisory_any_bar.py', WHEN,
       '--skills-dir', 'skills', '--backend', BACKEND, '--year', str(YEAR),
       '--no-live']
if MODEL.strip():
    cmd += ['--model', MODEL.strip()]
print('$', ' '.join(shlex.quote(c) for c in cmd), '\n')

proc = subprocess.run(cmd, capture_output=True, text=True, env=env)
print(proc.stderr)
print('=' * 70)
print(proc.stdout)
if proc.returncode != 0:
    print('\n[exit code]', proc.returncode)

## 6. The verbose log on this Colab VM
The full log is saved on the VM in `gdrive_tmp_<day>/` (re-runs accumulate as `_1/_2/_3`). The next cell sends it to the owner.

In [ ]:
import glob, os
logs = sorted(glob.glob(f'{DAY_FOLDER}/advisory_{req.yyyymmdd}_*.log'),
              key=os.path.getmtime)
if logs:
    print(f'{len(logs)} log(s) in {DAY_FOLDER}/:')
    for p in logs:
        print('   ', os.path.basename(p), f'({os.path.getsize(p)} bytes)')
    print('\n===== newest:', os.path.basename(logs[-1]), '=====\n')
    print(open(logs[-1], encoding='utf-8').read())
else:
    print('no log in', DAY_FOLDER, '— did the download or run fail above?')

## 7. Send this run's log to the OWNER (central collection)
POSTs your log to the owner's collector → saved to `external_user_logs/` on the **owner's** Drive as `<your name>_<date>_<time>.log` (`_1/_2/_3` on repeats). No Drive mount — just an HTTPS POST. Put your name in `EXTERNAL_USER` so the owner can tell whose run it is.

In [ ]:
#@title Send log to owner { run: "auto" }
EXTERNAL_USER = ""  #@param {type:"string"}

import glob, os
src_logs = sorted(glob.glob(f'{DAY_FOLDER}/advisory_{req.yyyymmdd}_*.log'),
                  key=os.path.getmtime)
if not src_logs:
    print('no log to send — run the previous cell first.')
elif not COLLECTOR_URL.startswith('http'):
    print('collector URL not set — your log is on the Colab VM:')
    print('   ', os.path.abspath(src_logs[-1]))
else:
    newest = src_logs[-1]
    who = (EXTERNAL_USER or 'anon').strip() or 'anon'
    body = {'user': who,
            'when': f'{req.yyyymmdd}_' + req.time.replace(':', ''),
            'log': open(newest, encoding='utf-8').read()}
    try:
        import requests
        resp = requests.post(COLLECTOR_URL, json=body, timeout=90).json()
        if resp.get('ok'):
            print('sent your log to the owner as:', resp.get('saved'))
        else:
            print('collector returned an error:', resp.get('error'))
    except Exception as e:
        print('could not reach collector:', type(e).__name__, e)
        print('your log is on the Colab VM:', os.path.abspath(newest))